#Bronze Layer

In [0]:
%run ./07_Pipeline_Audti_Logging

In [0]:
from datetime import datetime

start_time = datetime.now()

**FHIR TO DELTA**

### Schema Creation

In [0]:
%sql
SHOW SCHEMAS in db_healthcare_kl


In [0]:
rawSourcePath = "abfss://raw@shealthcarekl.dfs.core.windows.net/fhir/"

In [0]:
df = spark.read.option("multiline","true").json(rawSourcePath + "*.json")

df.show(5)

In [0]:
df.printSchema()

In [0]:
df.count()

### JSON TO DELTA

In [0]:
%sql
CREATE TABLE if not EXISTS db_healthcare_kl.bronze.fhir_raw
USING DELTA
LOCATION 'abfss://bronze@shealthcarekl.dfs.core.windows.net/fhir_raw';

In [0]:
%sql
SHOW TABLES IN db_healthcare_kl.bronze;

### Auto Loader

In [0]:
raw_path = "abfss://raw@shealthcarekl.dfs.core.windows.net/fhir"

schema_path = "abfss://bronze@shealthcarekl.dfs.core.windows.net/schema/fhir"

checkpoint_path = "abfss://bronze@shealthcarekl.dfs.core.windows.net/checkpoints/fhir"

bronze_output_path = "abfss://bronze@shealthcarekl.dfs.core.windows.net/fhir_raw"

In [0]:
schema = (
    spark.read
    .option("multiLine","true")
    .json(raw_path)
    .schema
)

In [0]:
from pyspark.sql.functions import current_timestamp

bronze_df = (
    spark.readStream
    .format("cloudFiles")
    .schema(schema)
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(raw_path)
    .withColumn(
        "ingestion_time",
        current_timestamp()
    )
)

In [0]:

bronze_df.writeStream\
        .format("delta")\
        .option(
            "checkpointLocation",
            checkpoint_path
        )\
        .trigger(availableNow=True)\
        .start(bronze_output_path)


In [0]:
bronze_table = spark.read.table("bronze.fhir_raw")

bronze_table.printSchema()

In [0]:
%sql
DESCRIBE DETAIL db_healthcare_kl.bronze.fhir_raw;

In [0]:
%sql
DESCRIBE HISTORY db_healthcare_kl.bronze.fhir_raw;

**HL7 to Delta**

In [0]:
%sql
DROP TABLE IF EXISTS db_healthcare_kl.bronze.hl7_adt_raw;

In [0]:
dbutils.fs.rm(
    "abfss://bronze@shealthcarekl.dfs.core.windows.net/checkpoints/hl7_adt",
    True
)

In [0]:
raw_path = "abfss://raw@shealthcarekl.dfs.core.windows.net/hl7/ADT"

checkpoint_path = "abfss://bronze@shealthcarekl.dfs.core.windows.net/checkpoints/hl7_adt"

bronze_output_path = "abfss://bronze@shealthcarekl.dfs.core.windows.net/hl7_adt_raw"

In [0]:
schema = (
    spark.read
    .text(raw_path)
    .schema
)

In [0]:
from pyspark.sql.functions import current_timestamp

bronze_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "binaryFile")
        .load(raw_path)
        .selectExpr(
            "path",
            "cast(content as string) as raw_message"
        )
        .withColumn("ingestion_timestamp", current_timestamp())
)

In [0]:
(
    bronze_df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .start(bronze_output_path)
        .awaitTermination()
)

In [0]:
%sql
CREATE TABLE db_healthcare_kl.bronze.hl7_adt_raw
USING DELTA
LOCATION 'abfss://bronze@shealthcarekl.dfs.core.windows.net/hl7_adt_raw';

In [0]:
%sql
DESCRIBE db_healthcare_kl.bronze.hl7_adt_raw;

In [0]:
df = spark.read.table("db_healthcare_kl.bronze.hl7_adt_raw")

df.select("path", "raw_message").show(truncate=False)

flat files

In [0]:
def ingest_to_bronze(table_name):
    
    raw_path = f"abfss://raw@shealthcarekl.dfs.core.windows.net/flat files/{table_name}"

    checkpoint_path = f"abfss://bronze@shealthcarekl.dfs.core.windows.net/checkpoints/{table_name}"

    bronze_output_path = f"abfss://bronze@shealthcarekl.dfs.core.windows.net/{table_name}_bronze"

    # Infer schema
    schema = (
        spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(raw_path)
            .schema
    )

    bronze_df = (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .schema(schema)
            .load(raw_path)
            .withColumn("ingestion_timestamp", current_timestamp())
    )

    (
        bronze_df.writeStream
            .format("delta")
            .option("checkpointLocation", checkpoint_path)
            .trigger(availableNow=True)
            .start(bronze_output_path)
            .awaitTermination()
    )

    print(f"Finished loading {table_name}")

In [0]:
tables = [
    "patients",
    "encounters",
    "conditions",
    "observations",
    "medications"
]

for tbl in tables:
    ingest_to_bronze(tbl)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS db_healthcare_kl.bronze.patients_raw
USING DELTA
LOCATION 'abfss://bronze@shealthcarekl.dfs.core.windows.net/patients_bronze';

CREATE TABLE IF NOT EXISTS db_healthcare_kl.bronze.encounters_raw
USING DELTA
LOCATION 'abfss://bronze@shealthcarekl.dfs.core.windows.net/encounters_bronze';

CREATE TABLE IF NOT EXISTS db_healthcare_kl.bronze.conditions_raw
USING DELTA
LOCATION 'abfss://bronze@shealthcarekl.dfs.core.windows.net/conditions_bronze';

CREATE TABLE IF NOT EXISTS db_healthcare_kl.bronze.observations_raw
USING DELTA
LOCATION 'abfss://bronze@shealthcarekl.dfs.core.windows.net/observations_bronze';

CREATE TABLE IF NOT EXISTS db_healthcare_kl.bronze.medications_raw
USING DELTA
LOCATION 'abfss://bronze@shealthcarekl.dfs.core.windows.net/medications_bronze';

In [0]:
end_time = datetime.now()

bronze_patient = spark.table('db_healthcare_kl.bronze.fhir_raw')
record_count =bronze_patient.count()


log_pipeline_run(
    pipeline_name="Healthcare Lakehouse",
    layer="Bronze",
    status="SUCCESS",
    records_processed=record_count,
    start_time=start_time,
    end_time=end_time
)